## How well does GPT 5 mini perform on the latter four binary tasks, jointly?

load data and prompts

In [7]:
# If needed:
# %pip install openai pandas openpyxl scikit-learn simpledorff python-dotenv

import os
import json
import time
import pandas as pd
import numpy as np

from dotenv import load_dotenv
from openai import OpenAI

import importlib
import binary_tasks_prompts

# --------------------------------------------------
# RELOAD PROMPTS
# --------------------------------------------------

importlib.reload(binary_tasks_prompts)

# Change ONLY this line to switch prompt condition
SYSTEM_PROMPT = binary_tasks_prompts.OPERATIONALIZED_PROMPT
PROMPT_NAME = "operationalized_binary_joint"

# Alternative:
# SYSTEM_PROMPT = frame_prompts.OPERATIONALIZED_PROMPT
# PROMPT_NAME = "operationalized_binary_joint"


# --------------------------------------------------
# MODEL SETTINGS
# --------------------------------------------------

MODEL = "gpt-5-mini"
REASONING_EFFORT = "medium"

load_dotenv()

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)


# --------------------------------------------------
# LOAD GOLD DATA
# --------------------------------------------------

df = pd.read_excel("frame_gold.xlsx")

required_columns = [
    "uuid",
    "title",
    "protest_excerpt",
    "demands_present",
    "protester_quote",
    "authority_quote",
    "thematic_present",
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

df = df[required_columns].copy()

df["title"] = (
    df["title"]
    .fillna("")
    .astype(str)
)

df["protest_excerpt"] = (
    df["protest_excerpt"]
    .fillna("")
    .astype(str)
)


# --------------------------------------------------
# NORMALIZE GOLD LABELS
# --------------------------------------------------

binary_cols = [
    "demands_present",
    "protester_quote",
    "authority_quote",
    "thematic_present",
]

for col in binary_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )


# --------------------------------------------------
# TEST FIRST
# --------------------------------------------------

test_df = df.head(3).copy()

RUN_DF = df

# When ready for all 120:
# RUN_DF = df

print(f"Model:  {MODEL}")
print(f"Prompt: {PROMPT_NAME}")
print(f"Running {len(RUN_DF)} samples.")

Model:  gpt-5-mini
Prompt: operationalized_binary_joint
Running 120 samples.


schemas, API call

In [10]:
# ============================================================
# STRICT OUTPUT SCHEMA
# ============================================================

BINARY_JOINT_SCHEMA = {
    "type": "object",
    "properties": {
        "demands_present": {
            "type": "string",
            "enum": ["yes", "no"],
        },
        "protester_quote": {
            "type": "string",
            "enum": ["yes", "no"],
        },
        "authority_quote": {
            "type": "string",
            "enum": ["yes", "no"],
        },
        "thematic_present": {
            "type": "string",
            "enum": ["yes", "no"],
        },
    },
    "required": [
        "demands_present",
        "protester_quote",
        "authority_quote",
        "thematic_present",
    ],
    "additionalProperties": False,
}


# ============================================================
# FORMAT EACH ARTICLE
# ============================================================

def format_input(title, excerpt):

    return f"""TITLE:
{title}

EXCERPT:
{excerpt}
"""


# ============================================================
# API CALL
# ============================================================

def annotate_binary_joint(
    title,
    excerpt,
    max_retries=3
):

    user_input = format_input(
        title,
        excerpt
    )

    for attempt in range(max_retries):

        try:

            response = client.responses.create(
                model=MODEL,

                instructions=SYSTEM_PROMPT,

                input=user_input,

                reasoning={
                    "effort": REASONING_EFFORT
                },

                text={
                    "verbosity": "low",
                    "format": {
                        "type": "json_schema",
                        "name": "binary_joint_annotation",
                        "strict": True,
                        "schema": BINARY_JOINT_SCHEMA,
                    },
                },
            )

            annotation = json.loads(
                response.output_text
            )


            # Token details
            cached_tokens = None
            reasoning_tokens = None

            try:
                cached_tokens = (
                    response.usage
                    .input_tokens_details
                    .cached_tokens
                )
            except Exception:
                pass

            try:
                reasoning_tokens = (
                    response.usage
                    .output_tokens_details
                    .reasoning_tokens
                )
            except Exception:
                pass


            return {
                **annotation,

                "response_id":
                    response.id,

                "input_tokens":
                    response.usage.input_tokens,

                "cached_tokens":
                    cached_tokens,

                "output_tokens":
                    response.usage.output_tokens,

                "reasoning_tokens":
                    reasoning_tokens,

                "error":
                    None,
            }


        except Exception as e:

            print(
                f"Attempt {attempt + 1}/{max_retries} failed: "
                f"{type(e).__name__}: {e}"
            )

            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)

            else:

                return {
                    "demands_present": None,
                    "protester_quote": None,
                    "authority_quote": None,
                    "thematic_present": None,

                    "response_id": None,
                    "input_tokens": None,
                    "cached_tokens": None,
                    "output_tokens": None,
                    "reasoning_tokens": None,

                    "error": str(e),
                }


# ============================================================
# RUN
# ============================================================

results = []

for count, (_, row) in enumerate(
    RUN_DF.iterrows(),
    start=1
):

    print(
        f"[{count}/{len(RUN_DF)}] "
        f"{row['uuid']}"
    )

    prediction = annotate_binary_joint(
        title=row["title"],
        excerpt=row["protest_excerpt"],
    )

    prediction["uuid"] = row["uuid"]

    results.append(prediction)


pred_df = pd.DataFrame(results)


# ============================================================
# RENAME MODEL OUTPUT COLUMNS
# ============================================================

pred_df = pred_df.rename(columns={
    "demands_present":
        "gpt_demands_present",

    "protester_quote":
        "gpt_protester_quote",

    "authority_quote":
        "gpt_authority_quote",

    "thematic_present":
        "gpt_thematic_present",
})


# ============================================================
# MERGE BACK ON UUID
# ============================================================

results_df = RUN_DF.merge(
    pred_df,
    on="uuid",
    how="left",
    validate="one_to_one"
)


# ============================================================
# ADD MODEL METADATA
# ============================================================

results_df["model"] = MODEL
results_df["prompt_name"] = PROMPT_NAME
results_df["reasoning_effort"] = REASONING_EFFORT


# ============================================================
# SAVE RAW PREDICTIONS IMMEDIATELY
# ============================================================

output_file = (
    f"frame_validation_results_"
    f"{PROMPT_NAME}_{MODEL}.xlsx"
)

results_df.to_excel(
    output_file,
    index=False
)

print(f"\nSaved: {output_file}")

print(
    f"Successful: "
    f"{results_df['error'].isna().sum()}"
)

print(
    f"Errors: "
    f"{results_df['error'].notna().sum()}"
)


# ============================================================
# TOKEN SUMMARY
# ============================================================

print(
    "\nAverage input tokens:",
    round(
        results_df["input_tokens"].mean(),
        1
    )
)

print(
    "Average cached tokens:",
    round(
        results_df["cached_tokens"].mean(),
        1
    )
)

print(
    "Average output tokens:",
    round(
        results_df["output_tokens"].mean(),
        1
    )
)

print(
    "Average reasoning tokens:",
    round(
        results_df["reasoning_tokens"].mean(),
        1
    )
)

display(
    results_df.head()
)

[1/120] d1334539-4cf3-4c5f-a0ef-88db5f167395
[2/120] b088e93f-1de4-4021-97bf-a90700b0a0ac
[3/120] fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad
[4/120] 155faae5-c8b6-441e-a20e-dce06e01cf32
[5/120] 50fea669-6828-44de-8a70-b448e398be11
[6/120] 81efb265-e3ba-4508-adc7-73cbc029f359
[7/120] 5fb219a8-9df7-4ad0-bc57-d46be3e327e0
[8/120] 252fb33d-5e6c-4cff-acc0-98c253f6c4b8
[9/120] 4acf5304-e74d-4a4f-bee0-caf65bd02abe
[10/120] 74fd09e2-35a3-495b-9b59-1610ec3398e1
[11/120] 7d9d161f-8087-4e0b-89e2-4a9bc1dcfd9d
[12/120] 0c5fa89f-f32f-4014-98e5-e8ad481b8a2f
[13/120] b2a2ef91-319a-47bf-9a61-1f8b4e192abf
[14/120] a92ef5b3-9891-4492-8a32-5024d7cc52dc
[15/120] c13350d6-ae7e-43f9-8999-17e530a4c80b
[16/120] 874c16ec-4684-40c7-8928-06baaa20260d
[17/120] 0192cd22-a165-4eb8-bf6d-da6165164b2e
[18/120] 95c2c809-f17e-4d9d-af74-26ba987929cd
[19/120] 109cd735-179b-4d53-ba32-ac1b70d5d990
[20/120] 36ab3319-2965-4075-80bd-14627327ab65
[21/120] f4bf7615-7d65-4e3d-b332-fbf590087f99
[22/120] 58a57f20-02b7-4389-9add-4cd5c76aae

,uuid,title,protest_excerpt,demands_present,protester_quote,authority_quote,thematic_present,gpt_demands_present,gpt_protester_quote,gpt_authority_quote,gpt_thematic_present,response_id,input_tokens,cached_tokens,output_tokens,reasoning_tokens,error,model,prompt_name,reasoning_effort
0,d1334539-4cf3-4c5f-a0ef-88db5f167395,Third politician in a month killed in Ecuador,The gang boss was moved out of a jail with lig...,yes,no,no,no,yes,yes,no,no,resp_00412bf269f8d418006a91dbad768087d191f0c62...,2584,0,360,320,None,gpt-5-mini,operationalized_binary_joint,medium
1,b088e93f-1de4-4021-97bf-a90700b0a0ac,Amazon CEO tells staff to work in office three...,This comes after Amazon said it would lay off ...,yes,yes,no,yes,yes,yes,no,yes,resp_0d05b79cd5a3b81a006a91dbb39f2487d1b4b0fd2...,2650,1792,675,640,None,gpt-5-mini,operationalized_binary_joint,medium
2,fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad,Gaza ceasefire protest in Capitol rotunda lead...,Capitol rotunda on Sept. 27. Photo: Michael Ro...,yes,no,yes,yes,yes,no,yes,no,resp_00007d755698b756006a91dbbcf41487d1815d134...,2642,1792,359,320,None,gpt-5-mini,operationalized_binary_joint,medium
3,155faae5-c8b6-441e-a20e-dce06e01cf32,"Daniel Penny, accused in Jordan Neely subway c...",Neely and could not have foreseen his untimely...,yes,yes,yes,no,yes,no,yes,no,resp_0271a2a75acee7c9006a91dbc2428887d18568267...,2586,1792,495,448,None,gpt-5-mini,operationalized_binary_joint,medium
4,50fea669-6828-44de-8a70-b448e398be11,Legal scholars raising constitutional argument...,Not all in the legal community agree – and wha...,no,no,yes,yes,yes,no,yes,yes,resp_0c0809491d778880006a91dbc8e8ac87d1aa93873...,2884,1792,727,640,None,gpt-5-mini,operationalized_binary_joint,medium


results and analysis

In [11]:
# ============================================================
# EVALUATION — FOUR BINARY VARIABLES
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import simpledorff


# ------------------------------------------------------------
# KRIPPENDORFF'S ALPHA
# ------------------------------------------------------------

def calculate_alpha(
    y_true,
    y_pred
):

    alpha_df = pd.DataFrame({
        "item_id":
            np.arange(len(y_true)).repeat(2),

        "annotator_id":
            ["human", "gpt"] * len(y_true),

        "label":
            np.column_stack([
                y_true.to_numpy(),
                y_pred.to_numpy()
            ]).reshape(-1)
    })

    return (
        simpledorff
        .calculate_krippendorffs_alpha_for_df(
            alpha_df,
            experiment_col="item_id",
            annotator_col="annotator_id",
            class_col="label"
        )
    )


# ------------------------------------------------------------
# TASK DEFINITIONS
# ------------------------------------------------------------

tasks = {
    "demands_present": {
        "gold": "demands_present",
        "pred": "gpt_demands_present",
    },

    "protester_quote": {
        "gold": "protester_quote",
        "pred": "gpt_protester_quote",
    },

    "authority_quote": {
        "gold": "authority_quote",
        "pred": "gpt_authority_quote",
    },

    "thematic_present": {
        "gold": "thematic_present",
        "pred": "gpt_thematic_present",
    },
}


# ------------------------------------------------------------
# CALCULATE METRICS
# ------------------------------------------------------------

summary_rows = []

for task_name, info in tasks.items():

    valid = results_df[
        [
            info["gold"],
            info["pred"]
        ]
    ].dropna()

    y_true = valid[
        info["gold"]
    ]

    y_pred = valid[
        info["pred"]
    ]


    summary_rows.append({
        "task":
            task_name,

        "n":
            len(valid),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "yes_precision":
            precision_score(
                y_true,
                y_pred,
                pos_label="yes",
                zero_division=0
            ),

        "yes_recall":
            recall_score(
                y_true,
                y_pred,
                pos_label="yes",
                zero_division=0
            ),

        "yes_f1":
            f1_score(
                y_true,
                y_pred,
                pos_label="yes",
                zero_division=0
            ),

        "kripp_alpha":
            calculate_alpha(
                y_true,
                y_pred
            ),
    })


summary_df = pd.DataFrame(
    summary_rows
)


metric_cols = [
    "accuracy",
    "macro_f1",
    "yes_precision",
    "yes_recall",
    "yes_f1",
    "kripp_alpha",
]

summary_df[
    metric_cols
] = (
    summary_df[
        metric_cols
    ]
    .round(3)
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

display(
    summary_df
)


# ------------------------------------------------------------
# SAVE SUMMARY
# ------------------------------------------------------------

summary_file = (
    f"summary_"
    f"{PROMPT_NAME}_{MODEL}.xlsx"
)

summary_df.to_excel(
    summary_file,
    index=False
)

print(
    f"Saved: {summary_file}"
)

,task,n,accuracy,macro_f1,yes_precision,yes_recall,yes_f1,kripp_alpha
0,demands_present,120,0.825,0.764,0.816,0.964,0.884,0.530
1,protester_quote,120,0.825,0.821,0.788,0.804,0.796,0.644
2,authority_quote,120,0.767,0.713,0.742,0.960,0.837,0.428
3,thematic_present,120,0.742,0.739,0.633,0.962,0.763,0.481


Saved: summary_operationalized_binary_joint_gpt-5-mini.xlsx
